# Scenario Analysis
Run scenario tests (demand increase, capacity reduction, cost change) and compare total allocation costs.
Results are saved to `outputs/scenario_results.csv`.

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os

# Import optimization function
from src.optimization_model import build_and_solve

ModuleNotFoundError: No module named 'src'

## Load original data
Read the base CSVs from `data/`.

In [ ]:
base = Path('..').resolve()
data_dir = base / 'data'
out_dir = base / 'outputs'
out_dir.mkdir(exist_ok=True, parents=True)
customers = pd.read_csv(data_dir / 'customers.csv')
warehouses = pd.read_csv(data_dir / 'warehouses.csv')
transport = pd.read_csv(data_dir / 'transport_cost.csv')
customers.head()

## Helper: run scenario
This function will write modified CSVs for a scenario, call the optimization, and return the summary.

In [ ]:
def run_scenario(name, customers_df=None, warehouses_df=None, transport_df=None):
    scenario_dir = out_dir / f'scenario_{name}'
    scenario_dir.mkdir(exist_ok=True, parents=True)

    # Use provided modified dfs or fall back to originals
    cust = customers_df.copy() if customers_df is not None else customers.copy()
    wh = warehouses_df.copy() if warehouses_df is not None else warehouses.copy()
    tr = transport_df.copy() if transport_df is not None else transport.copy()

    cust_fp = scenario_dir / 'customers.csv'
    wh_fp = scenario_dir / 'warehouses.csv'
    tr_fp = scenario_dir / 'transport_cost.csv'
    out_fp = scenario_dir / 'optimal_allocation.csv'

    cust.to_csv(cust_fp, index=False)
    wh.to_csv(wh_fp, index=False)
    tr.to_csv(tr_fp, index=False)

    res = build_and_solve(cust_fp, wh_fp, tr_fp, out_fp)
    # include scenario name and output file path
    return {'scenario': name, 'total_cost': res.get('total_cost'), 'status': res.get('status'), 'allocation_csv': res.get('output_path')}

## Baseline
Run optimization on the baseline data first.

In [ ]:
baseline = run_scenario('baseline')
baseline

## Scenario 1 — Demand increase
Increase all customer demands by 20% and re-run.

In [ ]:
cust_up = customers.copy()
cust_up['demand'] = (cust_up['demand'] * 1.2).round().astype(int)
scenario_demand = run_scenario('demand_increase_20pct', customers_df=cust_up)
scenario_demand

## Scenario 2 — Capacity reduction
Reduce all warehouse capacities by 30% and re-run.

In [ ]:
wh_down = warehouses.copy()
wh_down['capacity'] = np.floor(wh_down['capacity'] * 0.7).astype(int)
scenario_capacity = run_scenario('capacity_reduction_30pct', warehouses_df=wh_down)
scenario_capacity

## Scenario 3 — Cost increase
Increase transport costs by 25% and re-run.

In [ ]:
tr_up = transport.copy()
tr_up['cost_per_unit'] = (tr_up['cost_per_unit'] * 1.25)
scenario_cost = run_scenario('cost_increase_25pct', transport_df=tr_up)
scenario_cost

## Compare scenarios and export results
Collect results into a CSV `outputs/scenario_results.csv`.

In [ ]:
results = [baseline, scenario_demand, scenario_capacity, scenario_cost]
df_res = pd.DataFrame(results)
df_res
# Save to outputs/scenario_results.csv (root outputs folder)
summary_fp = out_dir / 'scenario_results.csv'
df_res.to_csv(summary_fp, index=False)
print('Wrote', summary_fp)